# Janus 0.3 (janus_03) – tanítás és kiértékelés egyben

**Egyszer kell beállítani, utána csak Save Version → Save & Run All.**

Jobb oldali panel:
1. **Accelerator** → GPU T4 x2
2. **Internet** → On
3. **Input → Add Input** → `asvpoof-2019-dataset` (awsaf49)

Ne a cellákat futtasd egyesével. Nyomd meg a jobb felső **Save Version** gombot, válaszd a **Save & Run All** módot, és zárd be a böngészőt. Kb. 1 óra múlva a verzió **Output** fülén ott lesz minden: `janus_best.pt`, `eval_scores.csv`, `eval_per_attack.csv`, és a **Logs** fülön a teljes kimenet.

Lépések: tanítás 30 epoch, ResNet-stílusú modell, LFCC bemenet (~30 perc) → kiértékelés az eval halmazon (~20 perc).

## 0. Környezet ellenőrzése

In [ ]:
import os, sys, torch
from pathlib import Path

print('torch', torch.__version__)
assert torch.cuda.is_available(), 'NINCS GPU. Jobb oldali panel → Accelerator → GPU T4 x2, majd Run All újra.'
print('GPU:', torch.cuda.get_device_name(0))

inputs = list(Path('/kaggle/input').iterdir())
print('/kaggle/input:', [p.name for p in inputs])
assert inputs, 'NINCS ADAT. Jobb oldali panel → Input → Add Input → "asvpoof-2019-dataset" (awsaf49), majd Run All újra.'

## 1. Kód behúzása GitHubról

In [ ]:
REPO = 'https://github.com/Csaszi05/janus_03.git'
if not os.path.exists('/kaggle/working/janus_03'):
    !git clone -q {REPO} /kaggle/working/janus_03
else:
    !cd /kaggle/working/janus_03 && git pull -q
%cd /kaggle/working/janus_03
!git log --oneline | head -3

In [ ]:
!pip install -q -r requirements.txt

## 2. Adat megkeresése és ellenőrzése

Az adathalmaz gyökere az a mappa, amelyben az `ASVspoof2019_LA_cm_protocols` könyvtár van. A cella megkeresi a `/kaggle/input` alatt, bárhogy is van becsomagolva.

In [ ]:
def find_dir(root, name, depth=8):
    # szélességi keresés, max 8 szint mélyen; nem járja be a flac fájlokat
    level = [Path(root)]
    for _ in range(depth):
        nxt = []
        for d in level:
            for p in d.iterdir():
                if not p.is_dir():
                    continue
                if p.name == name:
                    return p
                nxt.append(p)
        level = nxt
    return None

proto = find_dir('/kaggle/input', 'ASVspoof2019_LA_cm_protocols')
assert proto, 'Nem találom az ASVspoof2019_LA_cm_protocols mappát a /kaggle/input alatt.'
DATA_ROOT = str(proto.parent); print('DATA_ROOT  =', DATA_ROOT)
!ls {DATA_ROOT}

sys.path.insert(0, 'src')
from dataset import load_protocol
tr = load_protocol(DATA_ROOT, 'train'); dv = load_protocol(DATA_ROOT, 'dev')
print('train', len(tr), 'spoof arány', round(tr.y.mean(), 3))
print('dev  ', len(dv), 'spoof arány', round(dv.y.mean(), 3))
assert os.path.exists(tr.path[0]), f'A protokoll megvan, de a flac fájl nem: {tr.path[0]}'

## 3. Tanítás (Janus 0.3: ResNet-stílusú modell LFCC-n)

- A modell `JanusResNet`: 5 reziduális blokk, 16→32→64→128→256 csatorna, kb. 1,2 millió paraméter.
- Minden más ugyanaz, mint a janus_01 Run 4-ben (Janus 0.2): LFCC bemenet, maszkolásos augmentáció, cosine ütemező, 30 epoch.
- Az LFCC-cache a `/tmp`-be megy; az első epoch ezért lassú (~14 perc), a továbbiak gyorsak.
- 100 batch-enként köztes loss, epochonként egy összefoglaló sor. A `*` jelzi, ha új legjobb dev-EER-t mentett.

In [ ]:
!python src/train.py \
    --data_root {DATA_ROOT} \
    --epochs 30 \
    --augment \
    --sched cosine \
    --batch 64 \
    --lr 1e-3 \
    --workers 2 \
    --cache_dir /tmp/cache \
    --out /kaggle/working/out

## 4. Eredmény

In [ ]:
ck = torch.load('/kaggle/working/out/janus_best.pt', map_location='cpu', weights_only=False)
print('legjobb epoch:', ck['epoch'])
print('audio config:', ck['config']['audio'])
!ls -la /kaggle/working/out

## 5. Kiértékelés az eval halmazon

Az eval halmazban 13 olyan támadástípus van (A07–A19), amit a modell tanításkor nem látott. Ez az igazi mérőszám. A támadásonkénti bontás megmutatja, melyik fog ki a modellen.

In [ ]:
!python src/evaluate.py \
    --data_root {DATA_ROOT} \
    --checkpoint /kaggle/working/out/janus_best.pt \
    --split eval \
    --workers 2 \
    --cache_dir /tmp/cache \
    --out /kaggle/working/out

## 6. Összefoglaló

In [ ]:
import pandas as pd
per = pd.read_csv('/kaggle/working/out/eval_per_attack.csv')
per['eer_%'] = (per.eer * 100).round(2)
print(per[['system', 'n', 'eer_%']].to_string(index=False))
!ls -la /kaggle/working/out